# WakeRef — Ajouter le décodeur fp16 au repo (SANS ré-uploader le modèle)

Le modèle est déjà sur ton repo HF `almorelle/whisper-wakeref-jib-onnx`. Ce mini-notebook :
1. **télécharge** le décodeur fp32 depuis le repo (rapide, CDN HF),
2. le **convertit en fp16** (~2× plus rapide sur GPU pour les passes longues, fiable partout),
3. **re-pousse uniquement** `onnx/decoder_model_merged_fp16.onnx` dans le repo.

Pas de GPU, pas de ré-entraînement, pas d'upload depuis ta machine. ~2-3 min. Remplis ton **token Write**.

In [ ]:
!pip -q install -U onnx onnxconverter-common huggingface_hub
print('OK')

In [ ]:
REPO  = 'almorelle/whisper-wakeref-jib-onnx'
TOKEN = 'hf_colle_ton_token_Write'       # <- huggingface.co/settings/tokens (Write)

from huggingface_hub import hf_hub_download
# encodeur + décodeur fp32 depuis le repo (+ external data si présent).
paths = {}
for f in ['onnx/encoder_model.onnx', 'onnx/decoder_model_merged.onnx']:
    paths[f] = hf_hub_download(REPO, f, local_dir='dl')
    try:
        hf_hub_download(REPO, f + '_data', local_dir='dl'); print('external data :', f + '_data')
    except Exception:
        print('pas de external data pour', f, '(ok)')
enc = paths['onnx/encoder_model.onnx']
dec = paths['onnx/decoder_model_merged.onnx']
print('enc :', enc); print('dec :', dec)

In [ ]:
import onnx, os
from onnxconverter_common import float16
# fp16 COMPLET (encodeur + décodeur, keep_io_types=False) : tout le graphe en fp16 →
# aucune frontière fp32/fp16 (l'erreur venait de là : sortie fp16 attendue en fp32 dans
# la cross-attention). L'encodeur fp16 sort du fp16 que le décodeur fp16 consomme direct.
def to_fp16(src, out):
    m = onnx.load(src, load_external_data=True)
    m16 = float16.convert_float_to_float16(m, keep_io_types=False)
    onnx.save(m16, out)
    print(f'{out} : {os.path.getsize(out)/1e6:.0f} Mo')
to_fp16(enc, 'encoder_model_fp16.onnx')
to_fp16(dec, 'decoder_model_merged_fp16.onnx')

In [ ]:
from huggingface_hub import login, HfApi
login(token=TOKEN)                        # auth AVANT le push (bloquant)
api = HfApi()
for out in ['encoder_model_fp16.onnx', 'decoder_model_merged_fp16.onnx']:
    api.upload_file(path_or_fileobj=out, path_in_repo='onnx/' + out, repo_id=REPO)
    print('poussé : onnx/' + out)
print('->', f'https://huggingface.co/{REPO}')

## C'est fait
Le repo a maintenant `onnx/decoder_model_merged_fp16.onnx`. L'app est branchée pour
l'utiliser sur WebGPU. Recharge `/judge/voix`, **dictée libre (jib)**, **Précharger**, teste la vitesse.

> Si ~2× ne suffit pas pour du live, le plan suivant est de ré-entraîner sur **whisper-base** (3× plus léger encore).